# Fiber Photometry Session Explorer

High-level example for loading and exploring one processed fiber photometry session. Reusable analysis functions live in `src/`; this notebook demonstrates how to use them.

## 1. Configuration

Change these paths for your installation and session. Placeholder paths are used so the committed notebook is not tied to one computer.

In [ ]:
# ============================================================
# Configuration
# ============================================================

from pathlib import Path

# Root directory containing the photometry data
PHOTOMETRY_ROOT = Path(
    r"Z:\Photometry"
)

# Session identifiers
MOUSE = "DK21"
DATE = "230704"
RUN = 2

# Photometry channel
CHANNEL = 1


# ============================================================
# Automatically construct the processed-session path
# ============================================================

SESSION_DIR = (
    PHOTOMETRY_ROOT
    / MOUSE
    / f"{MOUSE}_{DATE}"
)

PROCESSED_SESSION = (
    SESSION_DIR
    / f"{MOUSE}-{DATE}-{RUN:03d}-processed.npz"
)

print("Mouse:", MOUSE)
print("Date:", DATE)
print("Run:", RUN)
print()
print("Session directory:")
print(SESSION_DIR)
print()
print("Processed session:")
print(PROCESSED_SESSION)

In [ ]:
# ============================================================
# Imports
# ============================================================

import sys

# Repository root
PROJECT_ROOT = Path(
    r"C:\Users\lutasa2\Documents\photometry-analysis"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

import numpy as np
import matplotlib.pyplot as plt
import pynapple as nap

import src.save_sessiondata
import src.pynapple_utils
import src.plotting

print("Project:", PROJECT_ROOT)
print("Processed session:", PROCESSED_SESSION)

## 2. Load the processed session

Processed `.npz` files allow downstream analysis without rerunning raw-data preprocessing.

In [ ]:
# ============================================================
# Load processed session
# ============================================================

session = src.save_sessiondata.load_session(
    PROCESSED_SESSION
)

print("Mouse:", session["mouse"])
print("Date:", session["date"])
print("Run:", session["run"])

data = src.pynapple_utils.session_to_pynapple(
    session
)

print("\nAvailable Pynapple data:")

for key in data:
    print("  ", key)

## 3. Session overview

Check recording coverage, sampling rates, and event counts.

In [ ]:
dff = data["dff_ch1"]
locomotion = data["locomotion"]

dff_dt = np.median(np.diff(dff.t))
locomotion_dt = np.median(np.diff(locomotion.t))

print("dF/F")
print("  start:", dff.t[0])
print("  end:", dff.t[-1])
print("  samples:", len(dff))
print("  sampling rate:", 1 / dff_dt, "Hz")

print("\nLocomotion")
print("  start:", locomotion.t[0])
print("  end:", locomotion.t[-1])
print("  samples:", len(locomotion))
print("  sampling rate:", 1 / locomotion_dt, "Hz")

print("\nLicks:", len(session["lick_times"]))
print("Visual cues:", len(session["cue_onset"]))
print("Solenoid events:", len(session["solenoid_onset"]))

## 4. Raw photometry

Inspect the raw 465 and 405 channels as a quality-control step.

In [ ]:
raw_465 = np.asarray(session[f"photometry_465_ch{CHANNEL}"], dtype=float)
raw_405 = np.asarray(session[f"photometry_405_ch{CHANNEL}"], dtype=float)
time_465 = np.asarray(session[f"photo_time_465_ch{CHANNEL}"], dtype=float)
time_405 = np.asarray(session[f"photo_time_405_ch{CHANNEL}"], dtype=float)

plt.figure(figsize=(12, 5))
plt.plot(time_465, raw_465, label="465")
plt.plot(time_405, raw_405, label="405", alpha=0.8)
plt.xlabel("Time (s)")
plt.ylabel("Photometry")
plt.title("Raw photometry")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Processed dF/F

View the processed dF/F trace. Raw 465 can remain useful as an alternative representation when reference correction is questionable.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(data["dff_ch1"].t, data["dff_ch1"].d, linewidth=0.8)
plt.axhline(0, linestyle="--")
plt.xlabel("Time (s)")
plt.ylabel("dF/F")
plt.title("Processed dF/F")
plt.tight_layout()
plt.show()

## 6. Behavioral signals

Inspect locomotion, licking, cue onset, and solenoid/reward events.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(data["locomotion"].t, data["locomotion"].d, linewidth=0.8)
axes[0].set_ylabel("Locomotion")
axes[0].set_title("Locomotion")

axes[1].eventplot(session["lick_times"], orientation="horizontal")
axes[1].set_ylabel("Licks")
axes[1].set_title("Lick events")

axes[2].eventplot(
    session["cue_onset"], orientation="horizontal",
    lineoffsets=1, linelengths=0.8
)
axes[2].eventplot(
    session["solenoid_onset"], orientation="horizontal",
    lineoffsets=0, linelengths=0.8
)
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(["Solenoid", "Cue"])
axes[2].set_xlabel("Time (s)")
axes[2].set_title("Task events")

plt.tight_layout()
plt.show()

## 7. Available plotting functions

Reusable event-aligned and trial-level plotting functions are implemented in `src/plotting.py`.

In [ ]:
import inspect

print("Plotting functions currently available:")
for name, obj in inspect.getmembers(src.plotting):
    if inspect.isfunction(obj) and obj.__module__ == "src.plotting":
        print("  ", name)

## 8. Raw 465 versus processed dF/F

Compare raw 465 fractional fluorescence with processed dF/F on a common time axis.

In [ ]:
glm_time = np.asarray(data["locomotion"].t, dtype=float)

raw_465_tsd = nap.Tsd(t=time_465, d=raw_465, time_units="s")
raw_465_on_glm = raw_465_tsd.interpolate(
    nap.Ts(t=glm_time, time_units="s")
)

raw_465_fractional = np.asarray(raw_465_on_glm.d, dtype=float)
f0 = np.nanmedian(raw_465_fractional)
raw_465_fractional = (raw_465_fractional - f0) / f0

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(glm_time, raw_465_fractional, linewidth=0.8)
axes[0].set_ylabel("Fractional 465")
axes[0].set_title("Raw 465 fractional fluorescence")

axes[1].plot(data["dff_ch1"].t, data["dff_ch1"].d, linewidth=0.8)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("dF/F")
axes[1].set_title("Processed dF/F")

plt.tight_layout()
plt.show()

## 9. Optional NeMoS preparation

If NeMoS is installed, prepare raw 465 as a broad session-scale component plus residual fluorescence. The slow component is retained and is not automatically interpreted as pure photobleaching.

In [ ]:
try:
    import src.nemos_analysis

    behavioral = src.nemos_analysis.prepare_behavioral_response(
        session=session,
        data=data,
        channel=CHANNEL,
        n_slow_basis=10
    )

    print("GLM samples:", len(behavioral["glm_time"]))
    print("Raw 465 SD:", np.nanstd(behavioral["y"]))
    print("Slow component SD:", np.nanstd(behavioral["slow_prediction"]))
    print("Residual SD:", np.nanstd(behavioral["residual"]))

except (ImportError, ModuleNotFoundError) as error:
    behavioral = None
    print("NeMoS analysis is unavailable in this environment.")
    print(error)

In [ ]:
if behavioral is not None:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    axes[0].plot(
        behavioral["glm_time"], behavioral["y"],
        alpha=0.35, label="Raw 465"
    )
    axes[0].plot(
        behavioral["glm_time"], behavioral["slow_prediction"],
        linewidth=2, label="Slow session component"
    )
    axes[0].set_ylabel("Fractional fluorescence")
    axes[0].set_title("Raw 465 and broad session component")
    axes[0].legend()

    axes[1].plot(
        behavioral["glm_time"], behavioral["residual"],
        linewidth=0.8
    )
    axes[1].axhline(0, linestyle="--")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Residual 465")
    axes[1].set_title("Residual fluorescence")

    plt.tight_layout()
    plt.show()

## 10. Summary

This notebook provides a high-level entry point for one processed photometry session:

1. Load a processed `.npz`.
2. Convert it to Pynapple objects.
3. Check sampling rates and event counts.
4. Inspect raw 465/405 photometry.
5. Inspect processed dF/F.
6. Inspect behavior and task events.
7. Discover reusable plotting functions.
8. Compare raw 465 with dF/F.
9. Optionally prepare the response for NeMoS modeling.

For routine preprocessing, use the command-line wrapper in `scripts/`; reusable implementation belongs in `src/`.